# NB_GLD_SRV_POI_MULTI_LANG

RECOMPUTE `lh_vv_gold.dbo.gld_srv_poi_multi_lang`. Grain `(poi_id, lang)`.
`source_name` lấy từ `slv_poi_entity_non_enrich` theo `poi_id`.
Partner: `description` = `bio`; khi `bio` null thì lấy `description` của `slv_enr_poi_info_multi_lang`.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
PARTNER = "partner"
ML_COLS = ("poi_id", "lang", "time_to_spend", "best_time", "cost_vnd", "description")
OUT_COLS = ("poi_id", "lang", "source_name", "time_to_spend", "best_time", "cost_vnd", "description")


def _require(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ConfigError(f"gld_srv_poi_multi_lang: {name} thiếu {missing}")


def build(ctx):
    ml = ctx.input("multi_lang", live_only=False)
    ent = ctx.input("entity", live_only=False)
    i18n = ctx.input("i18n", live_only=False)
    _require(ml, ML_COLS, "slv_enr_poi_info_multi_lang")
    _require(ent, ("poi_id", "source_name"), "slv_poi_entity_non_enrich")
    _require(i18n, ("business_service_id", "locale", "bio"), "slv_pn_business_service_i18ns")
    bio = i18n.select(uuid_md5(F.col("business_service_id"), F.lit(PARTNER)).alias("_pid"),
                      F.col("locale").alias("_loc"), "bio")
    partner = F.coalesce(F.lower(F.trim(F.col("source_name"))) == PARTNER, F.lit(False))
    rows = (ml.select(*ML_COLS).join(ent.select("poi_id", "source_name"), "poi_id", "left")
            .join(bio, partner & (F.col("poi_id") == F.col("_pid")) & (F.col("lang") == F.col("_loc")), "left"))
    ctx.warn("multi_lang_source_name_null", rows.where(F.col("source_name").isNull()).count())
    description = F.when(partner, F.coalesce(F.col("bio"), F.col("description"))).otherwise(F.col("description"))
    return rows.withColumn("description", description).select(*OUT_COLS)


SPEC = NodeSpec(
    trg_tbl="gld_srv_poi_multi_lang",
    inputs={"multi_lang": "slv_enr_poi_info_multi_lang", "entity": "slv_poi_entity_non_enrich",
            "i18n": "slv_pn_business_service_i18ns"},
    pk=("poi_id", "lang"),
    build=build,
    required_inputs=("multi_lang", "entity"),
    write_mode="MERGE",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
